# BB84 Quantum Key Distribution with Strict Single-Shot Contextual Error Audit

This notebook applies the same contextual-reference framework to a compact **BB84 quantum key distribution (QKD)** simulation.

The experiment explicitly separates

$$
\text{legitimate BB84 state preparation / basis evolution}
\neq
\text{contextual channel error}
\neq
\text{single-shot measurement deviation}.
$$

Each transmitted qubit is measured exactly once:

$$
\mathrm{SHOTS}=1.
$$

The clean and noisy probability distributions are retained only as simulation-side expected values for decomposition and audit.

A crucial QKD-specific boundary is enforced in the interpretation: the deterministic contextual-reference projection uses the ideal expected state associated with Alice's prepared bit and basis. In an actual secret-key protocol, Bob cannot be given Alice's undisclosed secret bit as a correction reference. Therefore the corrected key produced here is an **oracle/reference-recovery audit**, not a deployable BB84 post-processing method and not a security proof.

The operational BB84 result is the **raw noisy sifted key and its QBER**. The corrected result tests only whether the mathematical contextual-reference framework behaves consistently.

In [ ]:
!pip install -q pennylane pennylane-lightning numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 7.4 MB/s eta 0:00:00


## BB84 Protocol

Alice independently chooses a random bit

$$
a_i\in\{0,1\}
$$

and a random basis

$$
A_i\in\{Z,X\}.
$$

The four BB84 states are

$$
|0\rangle,\qquad |1\rangle,\qquad
|+\rangle=\frac{|0\rangle+|1\rangle}{\sqrt2},\qquad
|-\rangle=\frac{|0\rangle-|1\rangle}{\sqrt2}.
$$

Bob independently chooses

$$
B_i\in\{Z,X\}.
$$

After the quantum transmission, Alice and Bob publicly compare only their **basis choices**. Positions satisfying

$$
A_i=B_i
$$

form the sifted-key set.

For the raw noisy sifted key, the quantum bit error rate is

$$
\mathrm{QBER}
=
\frac{
\#\{i:A_i=B_i,\ b_i\neq a_i\}
}{
\#\{i:A_i=B_i\}
}.
$$

No eavesdropper is simulated in this notebook. Any raw QBER therefore originates from the explicitly modeled contextual channel error and single-shot measurement realization.

In [ ]:
import numpy as np
import pennylane as qml

SEED = 42
NOISE_SEED = SEED + 15015
SHOT_SEED = SEED + 909

N_TRANSMISSIONS = 256
SHOTS = 1
EPS = 1e-12

RX_ERROR_RANGE = (-0.060, 0.060)
RY_ERROR_RANGE = (-0.080, 0.080)
RZ_ERROR_RANGE = (-0.060, 0.060)

protocol_rng = np.random.default_rng(SEED)
noise_rng = np.random.default_rng(NOISE_SEED)
shot_rng = np.random.default_rng(SHOT_SEED)

alice_bits = protocol_rng.integers(0, 2, size=N_TRANSMISSIONS, dtype=int)
alice_bases = protocol_rng.integers(0, 2, size=N_TRANSMISSIONS, dtype=int)  # 0=Z, 1=X
bob_bases = protocol_rng.integers(0, 2, size=N_TRANSMISSIONS, dtype=int)

# One deterministic contextual coherent error triplet per transmission.
ERROR_MAP = np.zeros((N_TRANSMISSIONS, 3), dtype=float)
ERROR_MAP[:, 0] = noise_rng.uniform(*RX_ERROR_RANGE, size=N_TRANSMISSIONS)
ERROR_MAP[:, 1] = noise_rng.uniform(*RY_ERROR_RANGE, size=N_TRANSMISSIONS)
ERROR_MAP[:, 2] = noise_rng.uniform(*RZ_ERROR_RANGE, size=N_TRANSMISSIONS)

try:
    dev = qml.device("lightning.qubit", wires=1)
    BACKEND = "lightning.qubit"
except Exception:
    dev = qml.device("default.qubit", wires=1)
    BACKEND = "default.qubit"

print("=" * 88)
print("BB84 CONFIGURATION")
print("=" * 88)
print(f"Backend                    : {BACKEND}")
print(f"Transmitted qubits         : {N_TRANSMISSIONS}")
print(f"Shots per transmitted qubit: {SHOTS}")
print(f"Protocol seed              : {SEED}")
print(f"Noise seed                 : {NOISE_SEED}")
print(f"RX observed range          : [{ERROR_MAP[:,0].min():+.5f}, {ERROR_MAP[:,0].max():+.5f}]")
print(f"RY observed range          : [{ERROR_MAP[:,1].min():+.5f}, {ERROR_MAP[:,1].max():+.5f}]")
print(f"RZ observed range          : [{ERROR_MAP[:,2].min():+.5f}, {ERROR_MAP[:,2].max():+.5f}]")

BB84 CONFIGURATION
Backend                    : lightning.qubit
Transmitted qubits         : 256
Shots per transmitted qubit: 1
Protocol seed              : 42
Noise seed                 : 15057
RX observed range          : [-0.05978, +0.05958]
RY observed range          : [-0.07984, +0.07950]
RZ observed range          : [-0.05927, +0.05998]


## Legitimate Preparation, Contextual Channel Error, and Bob's Basis Rotation

For transmission $i$, Alice's legitimate state preparation is denoted by

$$
U_i^{A}(a_i,A_i).
$$

The contextual channel perturbation is

$$
E_i
=
R_Z(\epsilon_i^Z)
R_Y(\epsilon_i^Y)
R_X(\epsilon_i^X).
$$

Bob's legitimate measurement-basis transformation is $U_i^B(B_i)$. For an $X$-basis measurement this is a Hadamard before computational-basis measurement; for a $Z$-basis measurement it is the identity.

The clean expected trajectory is

$$
|\psi_i^{\mathrm{ideal}}\rangle
=
U_i^B U_i^A|0\rangle,
$$

whereas the noisy trajectory is

$$
|\psi_i^{\mathrm{noisy}}\rangle
=
U_i^B E_i U_i^A|0\rangle.
$$

This ordering matters. The error is introduced **after Alice's legitimate preparation and before Bob's legitimate basis transformation**, so the channel error is propagated through Bob's later basis operation.

The framework therefore does not classify Alice's bit encoding or Bob's basis choice as error.

In [ ]:
def prepare_alice(bit, basis):
    # basis: 0=Z, 1=X
    if bit == 1:
        qml.PauliX(wires=0)
    if basis == 1:
        qml.Hadamard(wires=0)

def apply_channel_error(index):
    ex, ey, ez = ERROR_MAP[index]
    qml.RX(ex, wires=0)
    qml.RY(ey, wires=0)
    qml.RZ(ez, wires=0)

def rotate_to_bob_basis(basis):
    if basis == 1:
        qml.Hadamard(wires=0)

@qml.qnode(dev)
def ideal_state(bit, alice_basis, bob_basis):
    prepare_alice(bit, alice_basis)
    rotate_to_bob_basis(bob_basis)
    return qml.state()

@qml.qnode(dev)
def noisy_state(bit, alice_basis, bob_basis, index):
    prepare_alice(bit, alice_basis)
    apply_channel_error(index)
    rotate_to_bob_basis(bob_basis)
    return qml.state()

def probs_from_state(state):
    p = np.abs(np.asarray(state, dtype=complex)) ** 2
    return np.asarray(p / p.sum(), dtype=float)

P_ideal = np.zeros((N_TRANSMISSIONS, 2), dtype=float)
P_noisy = np.zeros((N_TRANSMISSIONS, 2), dtype=float)

for i in range(N_TRANSMISSIONS):
    P_ideal[i] = probs_from_state(
        ideal_state(int(alice_bits[i]), int(alice_bases[i]), int(bob_bases[i]))
    )
    P_noisy[i] = probs_from_state(
        noisy_state(int(alice_bits[i]), int(alice_bases[i]), int(bob_bases[i]), i)
    )

print("Independent ideal and noisy BB84 expected trajectories generated.")

Independent ideal and noisy BB84 expected trajectories generated.


## Representative Stage-by-Stage Propagation Audit

For a representative transmission, the state is inspected at three contexts:

1. after Alice's legitimate state preparation,
2. after the contextual channel error,
3. after Bob's legitimate basis transformation.

The legitimate displacement at a stage is

$$
D_l^{\mathrm{legit}}
=
|\psi_l^{\mathrm{ideal}}\rangle-
|\psi_{l-1}^{\mathrm{ideal}}\rangle,
$$

while the same-context error is

$$
D_l^{\mathrm{error}}
=
|\psi_l^{\mathrm{noisy}}\rangle-
|\psi_l^{\mathrm{ideal}}\rangle.
$$

The final Bob-basis operation is applied to both branches. Thus a change produced by Bob's legitimate basis rotation remains legitimate evolution, while the pre-existing noisy displacement is propagated through that operation.

In [ ]:
audit_dev = qml.device("default.qubit", wires=1)

@qml.qnode(audit_dev)
def prepared_state(bit, basis):
    prepare_alice(bit, basis)
    return qml.state()

@qml.qnode(audit_dev)
def prepared_noisy_state(bit, basis, index):
    prepare_alice(bit, basis)
    apply_channel_error(index)
    return qml.state()

@qml.qnode(audit_dev)
def stateprep_then_bob(state, bob_basis):
    qml.StatePrep(state, wires=[0])
    rotate_to_bob_basis(bob_basis)
    return qml.state()

def align_phase(reference, state):
    reference = np.asarray(reference, dtype=complex)
    state = np.asarray(state, dtype=complex)
    z = np.vdot(reference, state)
    return state if abs(z) < EPS else state * np.conj(z / abs(z))

# Prefer a matched X-basis event so Bob's later H operation visibly propagates the error.
candidates = np.where((alice_bases == 1) & (bob_bases == 1))[0]
REP = int(candidates[0]) if len(candidates) else 0

bit = int(alice_bits[REP])
ab = int(alice_bases[REP])
bb = int(bob_bases[REP])

psi0 = np.array([1.0+0j, 0.0+0j])
prep_i = np.asarray(prepared_state(bit, ab), dtype=complex)
prep_n = align_phase(prep_i, np.asarray(prepared_noisy_state(bit, ab, REP), dtype=complex))

final_i = np.asarray(stateprep_then_bob(prep_i, bb), dtype=complex)
final_n = align_phase(final_i, np.asarray(stateprep_then_bob(prep_n, bb), dtype=complex))

prep_legit = float(np.sqrt(np.mean(np.abs(prep_i - psi0)**2)))
channel_error = float(np.sqrt(np.mean(np.abs(prep_n - prep_i)**2)))
bob_legit = float(np.sqrt(np.mean(np.abs(final_i - prep_i)**2)))
final_error = float(np.sqrt(np.mean(np.abs(final_n - final_i)**2)))
final_fidelity = float(np.abs(np.vdot(final_i, final_n))**2)

print("=" * 94)
print("REPRESENTATIVE BB84 STAGE-BY-STAGE PROPAGATION AUDIT")
print("=" * 94)
print(f"Transmission index                 : {REP}")
print(f"Alice bit                          : {bit}")
print(f"Alice basis                        : {'X' if ab else 'Z'}")
print(f"Bob basis                          : {'X' if bb else 'Z'}")
print(f"Legitimate preparation RMS         : {prep_legit:.8e}")
print(f"Channel contextual error RMS       : {channel_error:.8e}")
print(f"Legitimate Bob-basis evolution RMS : {bob_legit:.8e}")
print(f"Final same-context error RMS        : {final_error:.8e}")
print(f"Final ideal/noisy fidelity          : {final_fidelity:.10f}")
print("The channel error is present before Bob's basis operation and is propagated through it.")

REPRESENTATIVE BB84 STAGE-BY-STAGE PROPAGATION AUDIT
Transmission index                 : 0
Alice bit                          : 0
Alice basis                        : X
Bob basis                          : X
Legitimate preparation RMS         : 5.41196100e-01
Channel contextual error RMS       : 2.39157046e-02
Legitimate Bob-basis evolution RMS : 5.41196100e-01
Final same-context error RMS        : 2.39157046e-02
Final ideal/noisy fidelity          : 0.9988564053
The channel error is present before Bob's basis operation and is propagated through it.


## Strict Single-Shot Measurement and Residual Decomposition

Each transmitted qubit produces one binary measurement:

$$
m_i^{(1)}\in\{0,1\}.
$$

Equivalently, its one-hot measurement vector is

$$
M_i^{(1)}\in\{(1,0),(0,1)\}.
$$

For simulation audit only,

$$
\Delta_i^{\mathrm{context}}
=
E_i^{\mathrm{noisy}}
-
E_i^{\mathrm{ideal}},
$$

and

$$
S_i^{(1)}
=
M_i^{(1)}
-
E_i^{\mathrm{noisy}}.
$$

Therefore,

$$
R_i
=
M_i^{(1)}
-
E_i^{\mathrm{ideal}}
=
\Delta_i^{\mathrm{context}}
+
S_i^{(1)}.
$$

The decomposition closure is checked numerically over every transmission.

In [ ]:
def one_shot_vector(probs):
    outcome = int(shot_rng.choice(2, size=1, p=np.asarray(probs, dtype=float))[0])
    m = np.zeros(2, dtype=float)
    m[outcome] = 1.0
    return outcome, m

ideal_outcomes = np.zeros(N_TRANSMISSIONS, dtype=int)
noisy_outcomes = np.zeros(N_TRANSMISSIONS, dtype=int)
M_ideal_1 = np.zeros_like(P_ideal)
M_noisy_1 = np.zeros_like(P_noisy)

for i in range(N_TRANSMISSIONS):
    ideal_outcomes[i], M_ideal_1[i] = one_shot_vector(P_ideal[i])
    noisy_outcomes[i], M_noisy_1[i] = one_shot_vector(P_noisy[i])

context_expected = P_noisy - P_ideal
sampling_residual = M_noisy_1 - P_noisy
total_residual = M_noisy_1 - P_ideal
closure = total_residual - (context_expected + sampling_residual)

print("=" * 94)
print("STRICT SINGLE-SHOT BB84 RESIDUAL DECOMPOSITION")
print("=" * 94)
print(f"Shots per transmitted qubit          : {SHOTS}")
print(f"Context/hardware expected MAE        : {np.mean(np.abs(context_expected)):.8e}")
print(f"Single-shot sampling residual MAE    : {np.mean(np.abs(sampling_residual)):.8e}")
print(f"Total one-shot residual MAE          : {np.mean(np.abs(total_residual)):.8e}")
print(f"Residual decomposition closure max Δ : {np.max(np.abs(closure)):.8e}")

STRICT SINGLE-SHOT BB84 RESIDUAL DECOMPOSITION
Shots per transmitted qubit          : 1
Context/hardware expected MAE        : 1.06901191e-02
Single-shot sampling residual MAE    : 2.63006998e-01
Total one-shot residual MAE          : 2.61718750e-01
Residual decomposition closure max Δ : 5.55111512e-17


## Leakage-Audited Deterministic Reference Projection

The generic deterministic correction is

$$
R_i
=
M_i^{(1)}
-
E_i^{\mathrm{ideal}},
$$

$$
M_i^{\mathrm{corr}}
=
M_i^{(1)}-R_i
=
E_i^{\mathrm{ideal}}.
$$

The correction function receives no error-map entry, no injected rotation angle, no noise seed, and no noisy expected probability.

Its computational inputs are only

$$
M_i^{(1)}
\quad\text{and}\quad
E_i^{\mathrm{ideal}}.
$$

The correction percentage is calculated as

$$
C=
100\left(
1-
\frac{
\operatorname{MAE}(M^{\mathrm{corr}},E^{\mathrm{ideal}})
}{
\operatorname{MAE}(M^{(1)},E^{\mathrm{ideal}})
}
\right).
$$

### QKD-specific leakage warning

Absence of **error-injection leakage** is not equivalent to absence of **protocol-information leakage**.

For matched BB84 bases, Alice's ideal expected distribution identifies her encoded bit. Therefore supplying $E_i^{\mathrm{ideal}}$ for a secret-key transmission is effectively supplying information correlated with the secret bit.

Accordingly, the reference-projected result below is an **oracle audit of the correction mathematics only**. It must not be presented as an operational Bob-side BB84 correction method, a method for bypassing reconciliation/privacy amplification, or evidence of QKD security.

In [ ]:
def contextual_reference_correction(measured_one_shot, ideal_expected):
    # ERROR-INJECTION LEAKAGE BOUNDARY:
    # no ERROR_MAP, injected angles, NOISE_SEED, apply_channel_error(),
    # or P_noisy enters this function.
    residual = measured_one_shot - ideal_expected
    corrected = measured_one_shot - residual
    return corrected, residual

M_corrected = np.zeros_like(M_noisy_1)
correction_residual = np.zeros_like(M_noisy_1)

for i in range(N_TRANSMISSIONS):
    M_corrected[i], correction_residual[i] = contextual_reference_correction(
        M_noisy_1[i], P_ideal[i]
    )

pre_mae = float(np.mean(np.abs(M_noisy_1 - P_ideal)))
post_mae = float(np.mean(np.abs(M_corrected - P_ideal)))

if pre_mae > EPS:
    correction_percentage = 100.0 * (1.0 - post_mae / pre_mae)
else:
    correction_percentage = 100.0 if post_mae <= EPS else 0.0

print("=" * 94)
print("DETERMINISTIC CONTEXTUAL-REFERENCE CORRECTION AUDIT")
print("=" * 94)
print(f"Pre-correction → ideal MAE           : {pre_mae:.8e}")
print(f"Post-correction → ideal MAE          : {post_mae:.8e}")
print(f"ERROR CORRECTION PERCENTAGE          : {correction_percentage:.6f}%")
print()
print("CORRECTION INPUT BOUNDARY")
print("-" * 94)
print("Correction inputs                       : M_noisy_1, P_ideal")
print("ERROR_MAP passed to correction           : NO")
print("Injected RX/RY/RZ angles passed          : NO")
print("NOISE_SEED passed to correction          : NO")
print("P_noisy passed to correction             : NO (audit-only)")
print("Alice-reference information in P_ideal   : YES — oracle/reference audit")

DETERMINISTIC CONTEXTUAL-REFERENCE CORRECTION AUDIT
Pre-correction → ideal MAE           : 2.61718750e-01
Post-correction → ideal MAE          : 0.00000000e+00
ERROR CORRECTION PERCENTAGE          : 100.000000%

CORRECTION INPUT BOUNDARY
----------------------------------------------------------------------------------------------
Correction inputs                       : M_noisy_1, P_ideal
ERROR_MAP passed to correction           : NO
Injected RX/RY/RZ angles passed          : NO
NOISE_SEED passed to correction          : NO
P_noisy passed to correction             : NO (audit-only)
Alice-reference information in P_ideal   : YES — oracle/reference audit


## Sifting, Raw QBER, and Reference-Recovery Audit

The operationally meaningful simulated BB84 output is obtained from the **raw noisy one-shot outcomes**.

The sift mask is

$$
\mathcal S=\{i:A_i=B_i\}.
$$

The raw noisy QBER is

$$
\mathrm{QBER}_{\mathrm{raw}}
=
\frac{1}{|\mathcal S|}
\sum_{i\in\mathcal S}
\mathbf 1[m_i^{(1)}\neq a_i].
$$

For comparison, the deterministic reference projection can be converted back to a bit using

$$
\hat a_i^{\mathrm{ref}}
=
\arg\max_z M_i^{\mathrm{corr}}(z).
$$

For matched bases, this reference-projected bit should recover Alice's ideal encoded bit by construction. Its QBER is therefore reported only as an **audit/reference-recovery metric**, not as an operational QKD result.

A real BB84 implementation would instead use authenticated basis reconciliation, QBER estimation on disclosed test bits, error correction / information reconciliation, and privacy amplification. Those cryptographic post-processing stages are outside this demonstration.

In [ ]:
sift_mask = alice_bases == bob_bases
sift_indices = np.where(sift_mask)[0]

alice_sifted = alice_bits[sift_mask]
bob_raw_sifted = noisy_outcomes[sift_mask]

reference_bits = np.argmax(M_corrected, axis=1).astype(int)
reference_sifted = reference_bits[sift_mask]

ideal_single_sifted = ideal_outcomes[sift_mask]

raw_errors = int(np.sum(bob_raw_sifted != alice_sifted))
ideal_single_errors = int(np.sum(ideal_single_sifted != alice_sifted))
reference_errors = int(np.sum(reference_sifted != alice_sifted))

raw_qber = raw_errors / len(alice_sifted) if len(alice_sifted) else np.nan
ideal_single_qber = ideal_single_errors / len(alice_sifted) if len(alice_sifted) else np.nan
reference_qber = reference_errors / len(alice_sifted) if len(alice_sifted) else np.nan

print("=" * 94)
print("BB84 SIFTED-KEY / QBER AUDIT")
print("=" * 94)
print(f"Transmitted qubits                  : {N_TRANSMISSIONS}")
print(f"Sifted positions                    : {len(sift_indices)}")
print(f"Sifting fraction                    : {len(sift_indices)/N_TRANSMISSIONS:.6f}")
print(f"Raw noisy sifted-key errors         : {raw_errors}")
print(f"RAW NOISY QBER                      : {100*raw_qber:.6f}%")
print(f"Ideal single-shot sifted errors     : {ideal_single_errors}")
print(f"Ideal single-shot QBER              : {100*ideal_single_qber:.6f}%")
print(f"Reference-projected sifted errors   : {reference_errors}")
print(f"REFERENCE-PROJECTED QBER (AUDIT)    : {100*reference_qber:.6f}%")
print()
print("First 32 sifted bits:")
print("Alice      :", "".join(map(str, alice_sifted[:32])))
print("Bob raw    :", "".join(map(str, bob_raw_sifted[:32])))
print("Ref. audit :", "".join(map(str, reference_sifted[:32])))

BB84 SIFTED-KEY / QBER AUDIT
Transmitted qubits                  : 256
Sifted positions                    : 122
Sifting fraction                    : 0.476562
Raw noisy sifted-key errors         : 0
RAW NOISY QBER                      : 0.000000%
Ideal single-shot sifted errors     : 0
Ideal single-shot QBER              : 0.000000%
Reference-projected sifted errors   : 0
REFERENCE-PROJECTED QBER (AUDIT)    : 0.000000%

First 32 sifted bits:
Alice      : 00101110011110100110011111101100
Bob raw    : 00101110011110100110011111101100
Ref. audit : 00101110011110100110011111101100


## Final Interpretation

This BB84 notebook validates the same mathematical framework used in the other demonstrations while preserving the special security semantics of QKD.

The following statements are supported by the simulation:

$$
\boxed{
\text{legitimate protocol evolution is separated from contextual error}
}
$$

$$
\boxed{
\text{the contextual channel error is inserted before later Bob-basis evolution}
}
$$

$$
\boxed{
\text{single-shot sampling deviation is separated from expected contextual displacement}
}
$$

$$
\boxed{
\text{the residual decomposition is numerically auditable}
}
$$

and the correction function has no direct access to the injected error parameters.

However,

$$
\boxed{
\text{reference recovery}\neq\text{deployable QKD error correction}
}
$$

because the ideal contextual reference for a secret-key qubit contains information correlated with Alice's encoded bit. The corrected key is therefore an oracle/reference benchmark for the deterministic projection equation.

The scientifically appropriate operational QKD quantity in this notebook is the **raw noisy single-shot QBER**. The reference-projected QBER and correction percentage characterize the internal mathematical correction definition only.

In [ ]:
print("=" * 98)
print("FINAL BB84 + STRICT SINGLE-SHOT CONTEXTUAL CORRECTION SUMMARY")
print("=" * 98)
print(f"Transmitted qubits                       : {N_TRANSMISSIONS}")
print(f"Shots per transmitted qubit              : {SHOTS}")
print(f"Sifted key length                         : {len(alice_sifted)}")
print(f"Context/hardware expected MAE             : {np.mean(np.abs(context_expected)):.8e}")
print(f"Single-shot sampling residual MAE         : {np.mean(np.abs(sampling_residual)):.8e}")
print(f"Total one-shot residual MAE               : {np.mean(np.abs(total_residual)):.8e}")
print(f"Residual decomposition closure max Δ      : {np.max(np.abs(closure)):.8e}")
print(f"Corrected → ideal reference MAE            : {post_mae:.8e}")
print(f"ERROR CORRECTION PERCENTAGE                : {correction_percentage:.6f}%")
print(f"RAW NOISY QBER                             : {100*raw_qber:.6f}%")
print(f"REFERENCE-PROJECTED QBER (AUDIT ONLY)      : {100*reference_qber:.6f}%")
print()
print("LEAKAGE / INTERPRETATION BOUNDARY")
print("-" * 98)
print("Injection parameters passed to correction  : NONE")
print("Exact noisy expectation passed             : NO (audit-only)")
print("Ideal contextual reference available       : YES")
print("Ideal reference correlated with Alice bit  : YES — not operationally secret-safe")
print("Security claim                              : NONE")
print("=" * 98)

FINAL BB84 + STRICT SINGLE-SHOT CONTEXTUAL CORRECTION SUMMARY
Transmitted qubits                       : 256
Shots per transmitted qubit              : 1
Sifted key length                         : 122
Context/hardware expected MAE             : 1.06901191e-02
Single-shot sampling residual MAE         : 2.63006998e-01
Total one-shot residual MAE               : 2.61718750e-01
Residual decomposition closure max Δ      : 5.55111512e-17
Corrected → ideal reference MAE            : 0.00000000e+00
ERROR CORRECTION PERCENTAGE                : 100.000000%
RAW NOISY QBER                             : 0.000000%
REFERENCE-PROJECTED QBER (AUDIT ONLY)      : 0.000000%

LEAKAGE / INTERPRETATION BOUNDARY
--------------------------------------------------------------------------------------------------
Injection parameters passed to correction  : NONE
Exact noisy expectation passed             : NO (audit-only)
Ideal contextual reference available       : YES
Ideal reference correlated with Alice bit